# Exploratory Data Analysis: `all_2_prescriptions`

Source file: `all_2.csv`

This notebook performs a structured EDA: data loading, structure/dtypes, missing-value analysis, descriptive statistics, distributions, categorical breakdowns, and a correlation heatmap, finishing with a single combined dashboard figure saved as PNG.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=0.8)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

dataset_name = 'all_2_prescriptions'


## 1. Load data

In [2]:
csv_path = r"/mnt/user-data/uploads/all_2.csv"
try:
    df = pd.read_csv(csv_path, low_memory=False)
except UnicodeDecodeError:
    df = pd.read_csv(csv_path, low_memory=False, encoding='latin1')

print("Shape:", df.shape)
df.head()


Shape: (20194, 13)


,SEQN,RXDUSE,RXDDRUG,RXDDRGID,RXQSEEN,RXDDAYS,RXDRSC1,RXDRSC2,RXDRSC3,RXDRSD1,RXDRSD2,RXDRSD3,RXDCOUNT
0,73557,1,99999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
1,73557,1,INSULIN,d00262,2.0,1460.0,E11,NaN,NaN,Type 2 diabetes mellitus,NaN,NaN,2.0
2,73558,1,GABAPENTIN,d03182,1.0,243.0,G25.81,NaN,NaN,Restless legs syndrome,NaN,NaN,4.0
3,73558,1,INSULIN GLARGINE,d04538,1.0,365.0,E11,NaN,NaN,Type 2 diabetes mellitus,NaN,NaN,4.0
4,73558,1,OLMESARTAN,d04801,1.0,14.0,E11.2,NaN,NaN,Type 2 diabetes mellitus with kidney complicat...,NaN,NaN,4.0


## 2. Structure & data types

In [3]:
dtype_counts = df.dtypes.astype(str).value_counts()
print("Column count by dtype:")
print(dtype_counts)
print()
buf_info = []
df.info(buf=None)


Column count by dtype:
str        8
float64    3
int64      2
Name: count, dtype: int64

<class 'pandas.DataFrame'>
RangeIndex: 20194 entries, 0 to 20193
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   SEQN      20194 non-null  int64  
 1   RXDUSE    20194 non-null  int64  
 2   RXDDRUG   14100 non-null  str    
 3   RXDDRGID  14016 non-null  str    
 4   RXQSEEN   14019 non-null  float64
 5   RXDDAYS   14018 non-null  float64
 6   RXDRSC1   14022 non-null  str    
 7   RXDRSC2   714 non-null    str    
 8   RXDRSC3   127 non-null    str    
 9   RXDRSD1   13633 non-null  str    
 10  RXDRSD2   714 non-null    str    
 11  RXDRSD3   127 non-null    str    
 12  RXDCOUNT  14100 non-null  float64
dtypes: float64(3), int64(2), str(8)
memory usage: 2.0 MB


## 3. Missing value analysis

In [4]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print(f"Columns with missing values: {len(missing_df)} / {df.shape[1]}")
missing_df.head(30)


Columns with missing values: 11 / 13


,missing_count,missing_pct
RXDRSC3,20067,99.37
RXDRSD3,20067,99.37
RXDRSD2,19480,96.46
RXDRSC2,19480,96.46
RXDRSD1,6561,32.49
RXDDRGID,6178,30.59
RXQSEEN,6175,30.58
RXDDAYS,6176,30.58
RXDRSC1,6172,30.56
RXDDRUG,6094,30.18


## 4. Descriptive statistics (numeric columns)

In [5]:
numeric_cols_all = df.select_dtypes(include=np.number).columns.tolist()
desc = df[numeric_cols_all].describe().T
desc.head(30)


,count,mean,std,min,25%,50%,75%,max
SEQN,20194.0,78545.350946,2933.000334,73557.0,75984.25,78506.0,81063.0,83731.0
RXDUSE,20194.0,1.303110,0.469000,1.0,1.00,1.0,2.0,9.0
RXQSEEN,14019.0,1.165276,0.498539,1.0,1.00,1.0,1.0,3.0
RXDDAYS,14018.0,3917.636039,13874.344196,1.0,365.00,1095.0,2920.0,99999.0
RXDCOUNT,14100.0,5.907943,3.813666,1.0,3.00,5.0,8.0,23.0


## 5. Select representative columns for visualization

Given the potentially large number of columns, we pick the most complete (least missing) numeric columns and the most informative low-cardinality categorical columns for plotting.

In [6]:
id_like = ["SEQN", "RXDDRGID"]

# Numeric columns: exclude ID-like, exclude near-constant, rank by completeness then variance
num_candidates = [c for c in numeric_cols_all if c not in id_like]
completeness = df[num_candidates].notna().mean()
nunique = df[num_candidates].nunique()
num_candidates = [c for c in num_candidates if nunique.get(c, 0) > 1]
ranked_numeric = (completeness.loc[num_candidates]).sort_values(ascending=False)
plot_numeric_cols = ranked_numeric.head(12).index.tolist()
print("Numeric columns selected for plotting:")
print(plot_numeric_cols)

# Categorical-like columns: object/string dtype OR numeric with small nunique
obj_cols = df.select_dtypes(exclude=np.number).columns.tolist()
low_card_numeric = [c for c in numeric_cols_all if c not in id_like and 2 <= df[c].nunique() <= 10]
cat_candidates = [c for c in (obj_cols + low_card_numeric) if c not in id_like]
cat_candidates = [c for c in cat_candidates if df[c].nunique() <= 15 and df[c].nunique() >= 2]
plot_cat_cols = cat_candidates[:6]
print("Categorical columns selected for plotting:")
print(plot_cat_cols)


Numeric columns selected for plotting:
['RXDUSE', 'RXDCOUNT', 'RXQSEEN', 'RXDDAYS']
Categorical columns selected for plotting:
['RXDUSE', 'RXQSEEN']


## 6. Correlation heatmap

In [7]:
corr_cols = ranked_numeric.head(15).index.tolist()
plt.figure(figsize=(9, 7))
if len(corr_cols) >= 2:
    corr = df[corr_cols].corr()
    sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, linewidths=0.3, cbar_kws={'shrink': 0.7})
    plt.title(f'Correlation heatmap ({dataset_name})')
    plt.tight_layout()
else:
    plt.text(0.5, 0.5, 'Not enough numeric columns for correlation', ha='center')
plt.show()


## 7. Distributions of key numeric variables

In [8]:
n = len(plot_numeric_cols)
ncols = 4
nrows = int(np.ceil(n / ncols)) if n else 1
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3.5, nrows*2.8))
axes = np.array(axes).reshape(-1)
for i, c in enumerate(plot_numeric_cols):
    ax = axes[i]
    data = df[c].dropna()
    ax.hist(data, bins=30, color='#4C72B0', edgecolor='white')
    ax.set_title(c, fontsize=9)
for j in range(len(plot_numeric_cols), len(axes)):
    axes[j].axis('off')
fig.suptitle(f'Distributions of key numeric variables ({dataset_name})', y=1.02)
plt.tight_layout()
plt.show()


## 8. Categorical variable breakdowns

In [9]:
n = len(plot_cat_cols)
if n:
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*3))
    axes = np.array(axes).reshape(-1)
    for i, c in enumerate(plot_cat_cols):
        ax = axes[i]
        vc = df[c].value_counts(dropna=True).head(10)
        sns.barplot(x=vc.values, y=vc.index.astype(str), ax=ax, color='#55A868')
        ax.set_title(c, fontsize=9)
        ax.set_xlabel('')
    for j in range(len(plot_cat_cols), len(axes)):
        axes[j].axis('off')
    fig.suptitle(f'Categorical variable breakdowns ({dataset_name})', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No suitable low-cardinality categorical columns found.')


## 9. Combined EDA dashboard (single PNG canvas)

All the small plots above are combined into one figure/canvas and exported as a single PNG image for this dataset.

In [10]:
png_path = r"/mnt/user-data/outputs/all_2_EDA_dashboard.png"

n_num = len(plot_numeric_cols)
n_cat = len(plot_cat_cols)
hist_rows = int(np.ceil(n_num / 4)) if n_num else 1
cat_rows = int(np.ceil(n_cat / 3)) if n_cat else 0

total_rows = 2 + hist_rows + cat_rows  # row0: missing+corr, then hist_rows, then cat_rows
fig = plt.figure(figsize=(20, 5 + hist_rows*3.2 + cat_rows*3.2))
gs = gridspec.GridSpec(1 + hist_rows + cat_rows, 4, figure=fig, hspace=0.6, wspace=0.4)

# Row 0a: missingness (spans 2 cols)
ax_miss = fig.add_subplot(gs[0, 0:2])
if len(missing_df):
    top_missing = missing_df.head(15)
    sns.barplot(x=top_missing['missing_pct'], y=top_missing.index.astype(str), ax=ax_miss, color='#C44E52')
    ax_miss.set_title('Top missing-value columns (%)', fontsize=10)
else:
    ax_miss.text(0.5, 0.5, 'No missing values', ha='center')
    ax_miss.axis('off')

# Row 0b: correlation heatmap (spans 2 cols)
ax_corr = fig.add_subplot(gs[0, 2:4])
if len(corr_cols) >= 2:
    sns.heatmap(df[corr_cols].corr(), cmap='coolwarm', center=0, ax=ax_corr, cbar_kws={'shrink': 0.6})
    ax_corr.set_title('Correlation heatmap', fontsize=10)
    ax_corr.tick_params(labelsize=6)
else:
    ax_corr.text(0.5, 0.5, 'Not enough numeric cols', ha='center')
    ax_corr.axis('off')

# Histogram grid
for i, c in enumerate(plot_numeric_cols):
    r = 1 + i // 4
    cpos = i % 4
    ax = fig.add_subplot(gs[r, cpos])
    ax.hist(df[c].dropna(), bins=25, color='#4C72B0', edgecolor='white')
    ax.set_title(c, fontsize=8)
    ax.tick_params(labelsize=6)

# Categorical grid
for i, c in enumerate(plot_cat_cols):
    r = 1 + hist_rows + i // 3
    cpos_group = i % 3
    ax = fig.add_subplot(gs[r, cpos_group])
    vc = df[c].value_counts(dropna=True).head(8)
    sns.barplot(x=vc.values, y=vc.index.astype(str), ax=ax, color='#55A868')
    ax.set_title(c, fontsize=8)
    ax.tick_params(labelsize=6)
    ax.set_xlabel('')

fig.suptitle(f'Combined EDA Dashboard — all_2_prescriptions', fontsize=16, y=1.01)
fig.savefig(png_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved combined dashboard to:', png_path)


Saved combined dashboard to: /mnt/user-data/outputs/all_2_EDA_dashboard.png


## 10. Summary

This notebook covered: dataset shape and dtypes, missingness, descriptive statistics, univariate distributions of the most complete numeric variables, categorical breakdowns of low-cardinality fields, a correlation heatmap, and a single combined dashboard PNG summarizing all of the above.